# KrishiSetu AI 🌾🤖 - Google Colab GPU Training Pipeline

**Project**: KrishiSetu AI  
**Module**: ResNet18 Transfer Learning on PlantVillage Crop Disease Dataset  
**Target Hardware**: Free Google Colab NVIDIA GPU (T4 / V100)  

> **GPU Activation**: Navigate to `Runtime` -> `Change runtime type` -> Select `T4 GPU` -> `Save`.

---

### Step 1: Verify Hardware & GPU Acceleration

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not enabled! Please switch runtime to T4 GPU.")

### Step 2: Clone Repository & Install Dependencies

In [ ]:
# Clone the repository (or upload ml folder)
!git clone https://github.com/amoghvm/krishisetu-ai.git || echo "Repo already cloned"
%cd krishisetu-ai

# Install dependencies
!pip install -q -r ml/requirements.txt

### Step 3: Acquire & Extract PlantVillage Dataset
Downloads the official PlantVillage dataset containing 38 plant disease categories.

In [ ]:
import os
import urllib.request
import zipfile
from pathlib import Path

DATA_DIR = Path("data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Option A: Using public PlantVillage archive
ARCHIVE_URL = "https://github.com/spMohanty/PlantVillage-Dataset/archive/master.zip"
ZIP_PATH = Path("data/plantvillage_raw.zip")

if not any(DATA_DIR.iterdir()):
    print("Downloading PlantVillage dataset archive...")
    # Alternative: Using kagglehub or direct mirror
    !pip install -q kagglehub
    import kagglehub
    try:
        path = kagglehub.dataset_download("emmarex/plantdisease")
        print(f"Dataset downloaded to: {path}")
        import shutil
        src_dir = Path(path) / "PlantVillage"
        if not src_dir.exists():
            src_dir = Path(path)
        for item in src_dir.iterdir():
            if item.is_dir() and not item.name.startswith("."):
                dest = DATA_DIR / item.name
                if not dest.exists():
                    shutil.copytree(item, dest)
        print(f"Successfully populated {DATA_DIR} with classes.")
    except Exception as e:
        print(f"kagglehub fallback: {e}")
else:
    print(f"Dataset already present in {DATA_DIR}")

### Step 4: Dataset Inspection & Class Distribution

In [ ]:
from ml.dataset import inspect_dataset
import matplotlib.pyplot as plt

stats = inspect_dataset(str(DATA_DIR))
print(f"Total Images: {stats['total_images']}")
print(f"Total Classes: {stats['num_classes']}")

# Plot class distribution
plt.figure(figsize=(14, 6))
classes = list(stats['class_distribution'].keys())
counts = list(stats['class_distribution'].values())
plt.bar(range(len(classes)), counts, color="forestgreen")
plt.title("PlantVillage Class Distribution")
plt.xlabel("Class Index")
plt.ylabel("Image Count")
plt.xticks(range(len(classes)), [c.split('___')[-1] for c in classes], rotation=90, fontsize=8)
plt.tight_layout()
plt.savefig("models/class_distribution.png", dpi=150)
plt.show()

### Step 5: Execute ResNet18 Transfer Learning Training

In [ ]:
# Execute the training pipeline script on GPU
!python ml/train.py \
    --data-dir data/processed \
    --output-dir models \
    --epochs 10 \
    --batch-size 64 \
    --lr 0.001 \
    --device cuda \
    --patience 3

### Step 6: Comprehensive Model Evaluation & Metrics
Computes Top-1 Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and low-confidence distribution.

In [ ]:
!python ml/evaluate.py \
    --weights models/resnet18_plantvillage.pt \
    --mapping models/class_mapping.json \
    --data-dir data/processed \
    --output models/evaluation_report.json \
    --threshold 0.60 \
    --device cuda

### Step 7: Download Artifacts for Local Backend Inference
Packages the trained PyTorch weights and class metadata for transfer to the developer laptop.

In [ ]:
from google.colab import files
import zipfile

with zipfile.ZipFile("krishisetu_model_artifacts.zip", "w") as zf:
    zf.write("models/resnet18_plantvillage.pt", arcname="resnet18_plantvillage.pt")
    zf.write("models/class_mapping.json", arcname="class_mapping.json")
    zf.write("models/model_metadata.json", arcname="model_metadata.json")
    if Path("models/evaluation_report.json").exists():
        zf.write("models/evaluation_report.json", arcname="evaluation_report.json")

print("Downloading model artifacts zip...")
files.download("krishisetu_model_artifacts.zip")